# C4 · Captions and visual question answering with a small VLM (14 min)

**Question:** how useful and how reliable are a small generative VLM's answers?

Flow: caption Flickr images and score them against human captions with CLIPScore → counting questions on COCO128 with ground-truth counts (and a detector baseline) → yes/no existence questions to measure hallucination → live question panel.
On a GPU with 8 GB or more it uses SmolVLM2-2.2B; on smaller GPUs (e.g. 4 GB) SmolVLM2-500M. CPU works, very slowly. Outputs go to `outputs/c4/`.

In [ ]:
# Make the project folder importable (works from notebooks/ or the project root)
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
# ---- configuration -------------------------------------------------------
import torch
GPU_GB     = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
VLM_ID     = ("HuggingFaceTB/SmolVLM2-2.2B-Instruct" if GPU_GB >= 7     # needs ~5 GB in 16-bit
              else "HuggingFaceTB/SmolVLM2-500M-Video-Instruct")      # fits 4 GB GPUs (GTX 1650 etc.)
SPLIT_IMAGES = GPU_GB >= 7                             # tile images: more detail, ~10x slower per answer
CLIP_ID    = "openai/clip-vit-base-patch32"           # judge for CLIPScore
N_CAPTION  = 20
N_COUNT    = 30
N_EXIST    = 30                                        # images; 2 questions each (present + absent)
SEED       = 0

In [ ]:
import json, random, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from vlmkit.common import get_device, out_dir, save_json, show_grid, Timer
from vlmkit.clip_utils import Clip
from vlmkit.vlm import SmallVLM, parse_int, parse_yes_no
from vlmkit import data

DEVICE, OUT = get_device(), out_dir("c4")
random.seed(SEED)
with Timer() as t:
    vlm = SmallVLM(VLM_ID, device="cuda" if DEVICE == "cuda" else "cpu", split_images=SPLIT_IMAGES)
print(f"loaded {VLM_ID} on {vlm.device} ({vlm.dtype}) in {t.seconds:.0f} s")
clip = Clip(CLIP_ID, DEVICE)
flickr = data.load_flickr1k()
names, coco = data.coco128()
log = open(OUT / "answers.jsonl", "w", encoding="utf-8")

def ask(path, question, task, **extra):
    with Timer() as tm:
        answer = vlm.ask(path, question)
    rec = {"task": task, "image": str(path), "question": question, "answer": answer,
           "seconds": round(tm.seconds, 2), **extra}
    log.write(json.dumps(rec) + "\n"); log.flush()
    return rec

## 1. Captions scored with CLIPScore

CLIPScore here = 100 × cosine(image, caption) from CLIP, floored at 0. We score the model's caption and the five human captions on the same scale.

In [ ]:
cap_rows = []
for it in random.sample(flickr, N_CAPTION):
    rec = ask(it["path"], "Describe this image in one sentence.", "caption")
    im = Image.open(it["path"]).convert("RGB")
    human = [clip.score(im, c) for c in it["captions"]]
    cap_rows.append({"filename": it["filename"], "model_caption": rec["answer"],
                     "model_clipscore": clip.score(im, rec["answer"]),
                     "human_mean": float(np.mean(human)), "human_best": float(np.max(human)),
                     "seconds": rec["seconds"], "human_caption": it["captions"][0]})
cap = pd.DataFrame(cap_rows); cap.to_csv(OUT / "captions.csv", index=False)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(cap.human_mean, cap.model_clipscore, color="#4C78A8")
lo, hi = cap[["human_mean", "model_clipscore"]].min().min() - 1, cap[["human_mean", "model_clipscore"]].max().max() + 1
ax.plot([lo, hi], [lo, hi], ls="--", color="grey"); ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel("human captions (mean CLIPScore)"); ax.set_ylabel("model caption CLIPScore")
ax.set_title(f"Model above the line on {(cap.model_clipscore > cap.human_mean).mean():.0%} of images")
plt.tight_layout(); plt.savefig(OUT / "clipscore_scatter.png", dpi=120); plt.show()
print(f"mean seconds per caption: {cap.seconds.mean():.2f}")
cap[["model_caption", "human_caption", "model_clipscore", "human_mean"]].head(8)

In [ ]:
show_grid([Image.open(data.FLICKR_DIR / "images" / f) for f in cap.filename[:6]],
          [f"M: {m[:70]}\nH: {h[:70]}" for m, h in zip(cap.model_caption[:6], cap.human_caption[:6])],
          cols=3, size=4, save=OUT / "captions_examples.png")
print("High CLIPScore is not the same as correct: read the captions for invented details.")

## 2. Counting people: VLM vs ground truth vs a detector

COCO128 labels give the true number of people per image. YOLO11s gives a classic detector baseline.

In [ ]:
from ultralytics import YOLO
detector = YOLO("yolo11s.pt")

with_people = [it for it in coco if it["counts"].get("person", 0) > 0]
without = [it for it in coco if it["counts"].get("person", 0) == 0]
pick = random.sample(with_people, min(N_COUNT - 5, len(with_people))) + random.sample(without, min(5, len(without)))
count_rows = []
for it in pick:
    gt = it["counts"].get("person", 0)
    rec = ask(it["path"], "How many people are in this image? Answer with a number only.", "count", gt=gt)
    det = detector.predict(str(it["path"]), classes=[0], conf=0.25, verbose=False)[0]
    count_rows.append({"image": it["path"].name, "gt": gt, "vlm": parse_int(rec["answer"]),
                       "detector": len(det.boxes), "vlm_raw": rec["answer"]})
cnt = pd.DataFrame(count_rows); cnt.to_csv(OUT / "counting.csv", index=False)

def scores(col):
    ok = cnt[col].notna()
    err = (cnt.loc[ok, col] - cnt.loc[ok, "gt"]).abs()
    return {"answered": f"{ok.mean():.0%}", "exact": round((err == 0).mean(), 3),
            "within_1": round((err <= 1).mean(), 3), "MAE": round(err.mean(), 2)}
count_summary = pd.DataFrame({"VLM": scores("vlm"), "YOLO11s detector": scores("detector")}).T
print(count_summary)

# note: cnt["gt"], not cnt.gt - DataFrame.gt is pandas' "greater than" method
fig, ax = plt.subplots(figsize=(5.5, 5))
m = int(cnt[["gt", "vlm", "detector"]].max().max()) + 1
ax.plot([0, m], [0, m], ls="--", color="grey")
ax.scatter(cnt["gt"], cnt["detector"], label="detector", marker="s", alpha=0.7)
ax.scatter(cnt["gt"], cnt["vlm"], label="VLM", alpha=0.7)
ax.set_xlabel("true number of people"); ax.set_ylabel("predicted"); ax.legend()
ax.set_title("Counting: points on the dashed line are exact"); plt.tight_layout()
plt.savefig(OUT / "counting.png", dpi=120); plt.show()

In [ ]:
# Where the VLM miscounts: look at the images (crowds, small or partly hidden people?)
coco_paths = {it["path"].name: it["path"] for it in coco}
miss = cnt[cnt["vlm"] != cnt["gt"]].copy()
miss["err"] = (miss["vlm"].astype(float) - miss["gt"]).abs()
miss = miss.sort_values("err", ascending=False).head(8)
if len(miss):
    show_grid([Image.open(coco_paths[n]) for n in miss["image"]],
              [f"true {g}  VLM {v}  YOLO {d}" for g, v, d in zip(miss["gt"], miss["vlm_raw"], miss["detector"])],
              cols=4, size=3, save=OUT / "counting_errors.png")
else:
    print("The VLM counted every sampled image correctly.")

## 3. Existence questions: how often does the VLM say yes to something that is not there?

For each image: one question about a class that is present, one about a class that is absent (by COCO labels).

In [ ]:
all_names = list(names.values())
ex_rows = []
labelled = [c for c in coco if c["counts"]]
for it in random.sample(labelled, min(N_EXIST, len(labelled))):
    present = random.choice(list(it["counts"]))
    absent = random.choice([n for n in all_names if n not in it["counts"]])
    for cls, truth in [(present, True), (absent, False)]:
        rec = ask(it["path"], f"Is there a {cls} in this image? Answer yes or no.", "exist", truth=truth)
        ex_rows.append({"image": it["path"].name, "class": cls, "truth": truth,
                        "answer": parse_yes_no(rec["answer"]), "raw": rec["answer"]})
ex = pd.DataFrame(ex_rows); ex.to_csv(OUT / "existence.csv", index=False)
valid = ex[ex.answer.notna()].astype({"answer": bool, "truth": bool})
exist_summary = {
    "accuracy": round(float((valid.answer == valid.truth).mean()), 3),
    "hit_rate_on_present": round(float(valid[valid.truth].answer.mean()), 3),
    "hallucination_rate_on_absent": round(float(valid[~valid.truth].answer.mean()), 3),
    "unparsed_answers": int(ex.answer.isna().sum()),
}
log.close()
print(exist_summary)
ex[(ex.answer == True) & (ex.truth == False)].head(10)

In [ ]:
summary = {
    "vlm": VLM_ID, "device": vlm.device,
    "captions": {"n": len(cap), "model_clipscore_mean": round(cap.model_clipscore.mean(), 2),
                 "human_clipscore_mean": round(cap.human_mean.mean(), 2),
                 "seconds_per_answer": round(cap.seconds.mean(), 2)},
    "counting": count_summary.to_dict("index"),
    "existence": exist_summary,
}
save_json(summary, OUT / "summary.json")
summary

## 4. Try it live

Pick an image, ask the VLM anything, press **Run**. Try questions it should get right, then try to make it hallucinate ("What colour is the dog?" on a photo with no dog).

In [ ]:
from vlmkit.common import live

choices = {f"flickr / {it['filename']}": str(it["path"]) for it in flickr[:30]}
choices.update({f"coco128 / {it['path'].name}": str(it["path"]) for it in coco[:30]})

def vqa_panel(image, question, max_new_tokens):
    with Timer() as tm:
        answer = vlm.ask(image, question, max_new_tokens=max_new_tokens)
    show_grid([Image.open(image)], [f"Q: {question}"], cols=1, size=5)
    print(f"A: {answer}\n({tm.seconds:.1f} s on {vlm.device})")

live(vqa_panel, image=choices, question="What is happening in this picture? Mention anything unusual.",
     max_new_tokens=(8, 256, 8, 64))

## Exercises for students

1. Read the false "yes" answers above. Which absent classes does the model invent most, and why might that be (common co-occurrence)?
2. Change the counting prompt (for example "Count the people one by one, then give the total.") and re-run section 2. Does exact accuracy change?
3. **Transfer to your project:** use the existence check as a guardrail: before trusting a VLM incident report, ask yes/no about each object it mentions and compare with your detector.